In [4]:
import sys
sys.path.append('..')

from utils.prompts import render
from utils.llm_client import LLMClient
from utils.logging_utils import log_llm_call
from utils.router import pick_model
from utils.classified_output_util import save_classified_message


### Check for both input example

In [ ]:
examples = """
Example 1:
Message: Breaking News: Kelani River level at 9m.
Output: District: Colombo | Intent: Info | Priority: Low

Example 2:
Message: Kelani river rising near Colombo. People warned to evacuate low areas.
Output: District: Colombo | Intent: Info | Priority: Low

Example 3:
Message: SOS: 5 people trapped on a roof in Ja-Ela (Gampaha). Water rising fast. Need boat immediately.
Output: District: Gampaha | Intent: Rescue | Priority: High

Example 4:
Message: We are trapped on the roof with 3 kids!
Output: District: None | Intent: Rescue | Priority: High

Example 5:
Message: Donation drive starting at Town Hall. We need water bottles and biscuits.
Output: District: None | Intent: Donation | Priority: Low

Example 6:
Message: URGENT: Landslide in Kalutara. 12 people missing. Rescue team needed.
Output: District: Kalutara | Intent: Rescue | Priority: High
"""

loaded_messages = """
Breaking News: Kelani River level at 9m.
We are trapped on the roof with 3 kids!
"""

role_context = "a Disaster Management Triage Classifier"

model = pick_model('openai', 'general')
client = LLMClient('openai', model)

output = []

constraints = (
    "1. Categorize Intent as: Rescue, Supply, Info, Donation, or Other.\n"
    "2. Identify the District. If a specific landmark (like Kelani River) is mentioned, map it to its known District (e.g., Colombo).\n"
    "3. Use 'None' ONLY if no location or landmark is mentioned.\n"
    "4. Priority MUST be 'High' for immediate life threats (trapped, medical, landslides) and 'Low' for general news/info.\n"
    "5. Format output EXACTLY as: District: [Name] | Intent: [Type] | Priority: [Level]"
)

for msg in loaded_messages.strip().splitlines():
    msg = msg.strip()
    if not msg:
        continue

    # IMPORTANT: build a fresh prompt PER message (so query changes each time)
    prompt_text, spec = render(
        'few_shot.v1',
        role=role_context,
        examples=examples,
        query=f"Message: {msg}",
        constraints=constraints,
        format="District: {{district}} | Intent: {{intent}} | Priority: {{priority}}"
    )

    messages = [{'role': 'user', 'content': prompt_text}]
    response = client.chat(messages, temperature=0.0)  # more deterministic

    print(f'Message: "{msg}"')
    print('Response:')
    print(response['text'])
    print('---\n')
    
    log_llm_call('openai', model, 'few_shot', response['latency_ms'], response['usage'])

    output.append({'message': msg, 'response': response['text']})


Message: "Breaking News: Kelani River level at 9m."
Response:
District: Colombo | Intent: Info | Priority: Low
---

Message: "We are trapped on the roof with 3 kids!"
Response:
District: None | Intent: Rescue | Priority: High
---



In [ ]:
final_results = []

with open('..\\data\\Sample Messages.txt', 'r') as files:
    lines = files.readlines()
    
    for line in lines:
        message_text = line.strip()
        if not message_text: continue # Skip empty lines
        
        # Build prompt for each message (Tried to reuse but got inconsistent results)
        prompt_text_classi, spec = render(
            'few_shot.v1',
            role=role_context,
            examples=examples,
            query=f"Message: {message_text}",
            constraints=constraints,
            format="District: {{district}} | Intent: {{intent}} | Priority: {{priority}}"
        )
        
        messages_classi = [{'role': 'user', 'content': prompt_text_classi}]
        response_classi = client.chat(messages_classi, temperature=0.0)
        raw_output = response_classi['text'] # e.g., "District: Colombo | Intent: Info | Priority: Low"
        
        log_llm_call('openai', model, 'few_shot', response['latency_ms'], response['usage'])
        
        # print(raw_output)
        #     District: Colombo | Intent: Info | Priority: High
        #     District: Gampaha | Intent: Rescue | Priority: High
        #     District: Kandy | Intent: Info | Priority: Low

        # This splits by the "|" character and then by ":"
        try:
            parts = {
                item.split(":")[0].strip().lower(): item.split(":")[1].strip()
                for item in raw_output.split("|")
            }
            
            final_results.append({
                "district": parts.get("district", "None"),
                "intent": parts.get("intent", "Other"),
                "priority": parts.get("priority", "Low"),
            })
            
            # for test only, remove later
            # print(f"district: {parts.get('district', 'None')}, intent: {parts.get('intent', 'Other')}, priority: {parts.get('priority', 'Low')}")
            
        except Exception as e:
            print(f"Error parsing response for: {message_text} -> {e}")

# Save
for item in final_results:
    save_classified_message(
        district=item["district"],
        intent=item["intent"],
        priority=item["priority"]
    )

print("Success! File saved to output/classified_messages.xlsx")

Success! File saved to output/classified_messages.xlsx
